# 01 · Dataset overview

This notebook establishes the data facts and classification framing shared by the Logistic Regression and KNN tracks. The target is `diagnosis`; `id` is a record identifier and is excluded from candidate predictors. No model is fitted here.

**Proposed decision metric:** malignant-class recall, with benign/malignant labels shown explicitly in all reports. Missing a malignant case is the higher-cost error in this screening-oriented exercise. Precision, F1, and ROC-AUC are secondary diagnostics. This is a research exercise, not a clinical diagnostic tool.

**Validation plan for later modeling:** stratified partitions because records are non-temporal and the classes are imbalanced; all learned preprocessing must be fitted only on training partitions. Keep the test partition untouched until final evaluation.

**Known risks to audit:** modest sample size, correlated measurements, potential ID leakage, class imbalance, duplicates, missing/invalid values, and split sensitivity. A single split is not enough to establish robustness; use stratified cross-validation on training data when model-building begins.

In [1]:
from pathlib import Path
import random
import platform
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
DATA_PATH = Path('../data/raw/Raw_Breast_Cancer_Data.csv')
TARGET = 'diagnosis'
ID_COLUMN = 'id'
print(f'Python: {platform.python_version()}')
print(f'pandas: {pd.__version__} | NumPy: {np.__version__}')
print(f'Seed: {SEED}')

Python: 3.13.3
pandas: 3.0.5 | NumPy: 2.5.2
Seed: 42


In [2]:
df = pd.read_csv(DATA_PATH)
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]:,}')
print(f'DataFrame memory: {df.memory_usage(deep=True).sum() / 1024**2:.3f} MiB')
display(df.head())
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'unique': df.nunique(dropna=False)}))

Rows: 569 | Columns: 33
DataFrame memory: 0.166 MiB


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


,dtype,missing,unique
id,int64,0,569
diagnosis,str,0,2
radius_mean,float64,0,456
texture_mean,float64,0,479
perimeter_mean,float64,0,522
area_mean,float64,0,539
smoothness_mean,float64,0,474
compactness_mean,float64,0,537
concavity_mean,float64,0,537
concave points_mean,float64,0,542


In [3]:
print('Dtype counts:')
display(df.dtypes.astype(str).value_counts().rename('columns').to_frame())
print('Numeric summary:')
display(df.describe().T)
print('Target distribution:')
target_counts = df[TARGET].value_counts(dropna=False).rename_axis(TARGET).to_frame('count')
target_counts['fraction'] = target_counts['count'] / len(df)
display(target_counts)
print('ID unique:', df[ID_COLUMN].nunique(), 'of', len(df))

Dtype counts:


,columns
float64,31
int64,1
str,1


Numeric summary:


,count,mean,std,min,25%,50%,75%,max
id,569.0,3.037183e+07,1.250206e+08,8670.000000,869218.000000,906024.000000,8.813129e+06,9.113205e+08
radius_mean,569.0,1.412729e+01,3.524049e+00,6.981000,11.700000,13.370000,1.578000e+01,2.811000e+01
texture_mean,569.0,1.928965e+01,4.301036e+00,9.710000,16.170000,18.840000,2.180000e+01,3.928000e+01
perimeter_mean,569.0,9.196903e+01,2.429898e+01,43.790000,75.170000,86.240000,1.041000e+02,1.885000e+02
area_mean,569.0,6.548891e+02,3.519141e+02,143.500000,420.300000,551.100000,7.827000e+02,2.501000e+03
smoothness_mean,569.0,9.636028e-02,1.406413e-02,0.052630,0.086370,0.095870,1.053000e-01,1.634000e-01
compactness_mean,569.0,1.043410e-01,5.281276e-02,0.019380,0.064920,0.092630,1.304000e-01,3.454000e-01
concavity_mean,569.0,8.879932e-02,7.971981e-02,0.000000,0.029560,0.061540,1.307000e-01,4.268000e-01
concave points_mean,569.0,4.891915e-02,3.880284e-02,0.000000,0.020310,0.033500,7.400000e-02,2.012000e-01
symmetry_mean,569.0,1.811619e-01,2.741428e-02,0.106000,0.161900,0.179200,1.957000e-01,3.040000e-01


Target distribution:


,count,fraction
diagnosis,,
B,357,0.627417
M,212,0.372583


ID unique: 569 of 569


In [4]:
quality = pd.Series({
    'rows': len(df),
    'columns': df.shape[1],
    'missing_cells': int(df.isna().sum().sum()),
    'all_missing_columns': int(df.isna().all().sum()),
    'duplicate_rows': int(df.duplicated().sum()),
    'duplicate_ids': int(df[ID_COLUMN].duplicated().sum()),
    'non_numeric_predictor_columns': int(df.drop(columns=[TARGET, ID_COLUMN]).select_dtypes(exclude='number').shape[1]),
})
display(quality.to_frame('value'))
print('Target labels:', sorted(df[TARGET].dropna().unique().tolist()))
print('Non-positive numeric feature values (inspect context; zero can be valid):', int((df.drop(columns=[TARGET, ID_COLUMN]).select_dtypes(include='number') <= 0).sum().sum()))
empty_columns = df.columns[df.isna().all()].tolist()
print('All-missing columns to exclude from later feature matrices:', empty_columns)

,value
rows,569
columns,33
missing_cells,569
all_missing_columns,1
duplicate_rows,0
duplicate_ids,0
non_numeric_predictor_columns,0


Target labels: ['B', 'M']
Non-positive numeric feature values (inspect context; zero can be valid): 78
All-missing columns to exclude from later feature matrices: ['Unnamed: 32']


## Initial feasibility and risk assessment

The source CSV has 569 rows and 33 columns based on the initial inspection; this notebook recomputes those facts from the loaded file. That is enough to demonstrate regularized classical classifiers, but it is still a modest dataset: expect split-to-split uncertainty and avoid claims of clinical readiness. The target is binary and categorical. Use scaling inside a train-fitted pipeline for both Logistic Regression and distance-based KNN.

`id` must not be used as a feature. Correlation among the radius, perimeter, and area measurements (and between mean, standard-error, and worst measurements) is plausible and will be checked during visualization/modeling. The CSV includes an entirely empty `Unnamed: 32` column (569 missing cells); it carries no information and should be excluded from feature matrices while leaving the raw CSV unchanged. Duplicate rows and unusual values are quantified above before mitigation is considered. Zero is allowed for some shape measurements, so non-positive counts require feature-level interpretation rather than automatic row removal. There is no stated time column, so random stratification is the current validation proposal; this assumption should be revisited if row ordering or provenance reveals grouping or temporal structure.

### Self-critique

The strongest objection is that malignant recall as the sole primary metric ignores false positives and the dataset's sampling context; the dataset may not represent the prevalence or clinical costs of deployment. It is selected because this project is explicitly framed around breast-cancer classification and missed malignant cases warrant attention, but this does not make a model clinically useful. Later reports must show precision, confusion matrices, uncertainty/stability, and clearly state this limitation.